In [1]:
import numpy as np

import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import matplotlib.pyplot as plt

from EPOriGabors import EPGabors

ModuleNotFoundError: No module named 'EPOriGabors'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

I found that I needed a lot more convoluitonal layers than expected.

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, nout=2, input_size=64):
        super(ImprovedCNN, self).__init__()

        # Convolutional layers with batch normalization
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, padding=2)  # Larger kernel for better feature capture
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # Ensure consistent size
        self.dropout = nn.Dropout(0.5)

        # Calculate the size after convolutions
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, nout)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 64x64 -> 32x32

        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 32x32 -> 16x16

        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 16x16 -> 8x8

        # Conv block 4
        x = F.relu(self.bn4(self.conv4(x)))  # 8x8 -> 8x8
        x = self.adaptive_pool(x)  # 8x8 -> 4x4

        # Flatten and fully connected layers
        x = x.view(x.size(0), -1)  # flatten: 256*4*4
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64, 64)),  # Larger input size for better detail capture
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((64, 64)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets with different transforms for train/test
img_dir = 'images/'

# Load full dataset first to check class distribution
full_dataset = EPGabors(
    img_dir=img_dir,
    include_vertical=False,
    include_single=True,
    include_zerovar=True,
    filter_ss=[16, 32],
    filter_var=[4, 8],
    transform=test_transform  # Use test transform for initial loading
)

print(f"Total dataset size: {len(full_dataset)}")

In [ ]:
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_indices, test_indices = random_split(range(len(full_dataset)), [train_size, test_size])

# Create separate datasets with different transforms
class TransformDataset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices.indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        # Get the original image without transform
        original_transform = self.base_dataset.transform
        self.base_dataset.transform = None
        image, labels = self.base_dataset[actual_idx]
        self.base_dataset.transform = original_transform

        # Apply our transform
        if self.transform:
            image = self.transform(image)

        return image, labels

train_dataset = TransformDataset(full_dataset, train_indices, train_transform)
test_dataset = TransformDataset(full_dataset, test_indices, test_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

Found that having a scheduler helped. This basically will change the learning rate after each 10 epochs.

In [ ]:
model = ImprovedCNN(nout=2, input_size=64).to(device)

# Use CrossEntropyLoss with class weights to handle potential class imbalance
class_weights = torch.tensor([1.0, 1.0]).to(device)  # Adjust if needed based on distribution
lossfn = nn.CrossEntropyLoss(weight=class_weights)

# Use different learning rates and add scheduler
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)

In [ ]:
epochs = 30
train_losses = []

print("Starting training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        targets = (labels['mean'] > 0).long().to(device)  # CW = 1, CCW = 0

        optimizer.zero_grad()
        outputs = model(images)
        loss = lossfn(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Step scheduler
    scheduler.step()

    # Calculate average loss
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Simple training progress
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")


In [ ]:
print("\nFinal evaluation...")
model.eval()
correct = 0
total = 0
all_predictions = []
all_targets = []
all_mean_orientations = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        targets = (labels['mean'] > 0).long().to(device)
        mean_orientations = labels['mean'].cpu().numpy()

        outputs = model(images)
        predicted = torch.argmax(outputs, dim=1)

        correct += (predicted == targets).sum().item()
        total += targets.size(0)

        all_predictions.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_mean_orientations.extend(mean_orientations)

final_accuracy = 100 * correct / total
print(f"Final Test Accuracy: {final_accuracy:.2f}%")

In [ ]:
plt.figure(figsize=(12, 5))

# Convert to numpy arrays
all_mean_orientations = np.array(all_mean_orientations)
all_predictions = np.array(all_predictions)

# Sort by mean orientation for better visualization
sort_idx = np.argsort(all_mean_orientations)
sorted_means = all_mean_orientations[sort_idx]
sorted_preds = all_predictions[sort_idx]

# Plot 1: Prediction vs actual mean orientation scatter
plt.subplot(1, 2, 1)
plt.scatter(sorted_means, sorted_preds, alpha=0.6, s=15, c=sorted_preds, cmap='coolwarm')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='True Decision Boundary')
plt.xlabel("Actual Mean Orientation")
plt.ylabel("Predicted Class (0=CCW, 1=CW)")
plt.title("Predicted Class vs Actual Mean Orientation")
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 2: Show prediction transition as a function of mean orientation
plt.subplot(1, 2, 2)
# Create bins to show how predictions change across orientation values
bin_edges = np.linspace(sorted_means.min(), sorted_means.max(), 20)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_predictions = []

for i in range(len(bin_edges) - 1):
    mask = (sorted_means >= bin_edges[i]) & (sorted_means < bin_edges[i + 1])
    if np.sum(mask) > 0:
        bin_predictions.append(np.mean(sorted_preds[mask]))
    else:
        bin_predictions.append(np.nan)

plt.plot(bin_centers, bin_predictions, 'o-', linewidth=2, markersize=6, color='blue', label='Average Prediction')
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='Chance Level')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='True Decision Boundary')
plt.xlabel("Actual Mean Orientation")
plt.ylabel("Average Predicted Class")
plt.title("Prediction Probability vs Mean Orientation")
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()
